# **Diplomatura en Ciencia de Datos, Aprendizaje Automático y sus Aplicaciones**

## **Edición 2026**


----

# Trabajo práctico entregable - parte 1


Trabajaremos con la base de datos de `melb_data` presentada a continuación.

In [8]:
import matplotlib.pyplot as plt
import numpy
import pandas
import seaborn
seaborn.set_context('talk')

In [9]:
# Cargamos los datos
melb_df = pandas.read_csv(
    'https://cs.famaf.unc.edu.ar/~mteruel/datasets/diplodatos/melb_data.csv')
melb_df[:3]

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0


### Información acerca de las variables
Suburb: Suburb

Address: Dirección

Rooms: Número de habitaciones

Price: Precio en dólares australianos

Method:
S - propiedad vendida;
SP - propiedad vendida previamente;
PI - propiedad no vendida (pasó en subasta sin alcanzar el precio de reserva);
PN - vendida previamente no divulgada;
SN - vendida no divulgada;
NB - sin oferta;
VB - oferta del vendedor;
W - retirada antes de la subasta;
SA - vendida después de la subasta;
SS - vendida después de la subasta (precio no divulgado).
N/A - precio o oferta más alta no disponible.

Type:
br - dormitorio(s);
h - casa, cabaña, villa, semi-adosado, terraza;
u - unidad, dúplex;
t - casa adosada;
dev site - sitio de desarrollo;
o res - otra residencia.

SellerG: Agente inmobiliario

Date: Fecha de venta

Distance: Distancia al CBD en kilómetros

Regionname: Región general (Oeste, Noroeste, Norte, Noreste, etc.)

Propertycount: Número de propiedades existentes en el suburbio.

Bedroom2 : Número de dormitorios (obtenido de otra fuente)

Bathroom: Número de baños

Car: Número de plazas de aparcamiento

Landsize: Tamaño del terreno en metros cuadrados

BuildingArea: Superficie construida en metros cuadrados

YearBuilt: Año de construcción de la casa

CouncilArea: Área del consejo municipal

Lattitude: Latitud

Longtitude: Longitud


In [10]:
melb_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         13580 non-null  str    
 1   Address        13580 non-null  str    
 2   Rooms          13580 non-null  int64  
 3   Type           13580 non-null  str    
 4   Price          13580 non-null  float64
 5   Method         13580 non-null  str    
 6   SellerG        13580 non-null  str    
 7   Date           13580 non-null  str    
 8   Distance       13580 non-null  float64
 9   Postcode       13580 non-null  float64
 10  Bedroom2       13580 non-null  float64
 11  Bathroom       13580 non-null  float64
 12  Car            13518 non-null  float64
 13  Landsize       13580 non-null  float64
 14  BuildingArea   7130 non-null   float64
 15  YearBuilt      8205 non-null   float64
 16  CouncilArea    12211 non-null  str    
 17  Lattitude      13580 non-null  float64
 18  Longtitude     13

## Ejercicio 1: Encoding

1. Seleccionar todas las filas y columnas del conjunto de datos, **excepto** `BuildingArea` y `YearBuilt`.

2. Decidir si elimina algunas filas o columnas en base al análisis de datos faltantes.

3. Hacer un análisis descriptivo de las variables numéricas del conjunto de datos. Todas tienen el tipo `Dtype` correcto asignado? Armar una matriz (array) sólo con las variables numéricas.  

4. Estudiar las variables categóricas del DataFrame. Aplicar una codificación One-hot encoding a las columnas categóricas que crea pertinente. Si lo consideran necesario, pueden reducir el número de categorías únicas de algunas variables. ¿Cómo trataría la variable `Date`? Armar la matriz de variables categóricas codificada.

5. Concatenar la matriz de variables numéricas a la matriz que codifica las variables categóricas resultante del punto anterior.

Algunas opciones:
  1. Utilizar `OneHotEncoder` junto con el parámetro `categories` para las variables categóricas y luego usar `numpy.hstack` para concatenar el resultado con las variables numéricas.
  2. `DictVectorizer` con algunos pasos de pre-proceso previo.

Recordar también que el atributo `pandas.DataFrame.values` permite acceder a la matriz de numpy subyacente a un DataFrame.


### Ejercicio 1.1 — Subset sin `BuildingArea` ni `YearBuilt`

**Qué pide la consigna**: "Seleccionar todas las filas y columnas del conjunto de datos, **excepto** `BuildingArea` y `YearBuilt`".

Parece trivial: un `drop`. Pero atrás de ese `drop` hay una **decisión estratégica de pipeline** que conviene entender, porque define el orden de todo lo que sigue.

#### Por qué se excluyen JUSTO estas dos columnas

Mirá el `info()` de arriba: `BuildingArea` tiene 7.130 valores no-nulos sobre 13.580 filas (~47% faltantes) y `YearBuilt` tiene 8.205 (~40% faltantes). El resto de las columnas, salvo `Car` (62 NaN, 0.46%) y `CouncilArea` (1.369 NaN, 10.1%), está completo.

Si imputáramos `BuildingArea` con la mediana ahora — antes de hacer encoding de las categóricas — nos perdemos la chance de usar **Suburb, Type, Regionname, CouncilArea, etc.** como predictores de la imputación. Y son justamente los predictores más informativos: el precio y el tamaño construido de una casa dependen fuertemente del barrio y del tipo.

#### La estrategia del TP (que tenés que entender, no solo seguir)

El TP nos lleva por un camino concreto que vale la pena explicitar:

1. **Ahora** (Ej 1.1): dropeamos `BuildingArea` y `YearBuilt` del DataFrame de trabajo.
2. **Ej 1.2-1.5**: limpiamos el resto, encodeamos categóricas, armamos una matriz numérica densa.
3. **Ej 2.1**: re-agregamos `BuildingArea` y `YearBuilt` a esa matriz (todavía con NaN).
4. **Ej 2.2**: corremos `IterativeImputer` (MICE) con `KNeighborsRegressor` sobre la matriz completa. Como ahora tenemos toda la info categórica encodeada disponible, el imputador tiene un montón de variables para predecir esos NaN con calidad real.

Es decir: **postergamos** la imputación hasta tener la información completa que la haga buena. Si imputáramos antes, estaríamos prediciendo con menos información, y la imputación arrastraría sesgos que después PCA amplifica.

#### Conexión con los apuntes

- En `02-datos-faltantes.md` vimos que MICE/KNN se beneficia MUCHO de tener variables correlacionadas como predictores. Acá estamos construyendo justamente ese contexto.
- En el apunte `04-tipos-de-variables-y-encodings.md` está la cita en mayúsculas de la cátedra: *"ANTES DE LLEGAR ACÁ TENGO QUE HABER TRABAJADO CON MIS DATOS FALTANTES"*. Pero hay una sutileza: aplica a las columnas que vas a encodear ahora. `BuildingArea` y `YearBuilt` quedan **fuera** del encoding precisamente por eso, y se imputan en una segunda pasada.

#### Trampa típica

Confundir "trabajar con los faltantes antes de encodear" con "imputar todo de una con la media". Lo primero significa **decidir qué hacés con cada faltante**, no necesariamente imputar. Para `BuildingArea` y `YearBuilt` la decisión es: **postergar imputación, hacerla con un modelo serio después**. Eso ES trabajar con los faltantes.

#### Sobre el `.copy()` que vas a ver

`drop()` sin `inplace=True` **ya devuelve un objeto nuevo** — no muta `melb_df`. ¿Entonces para qué `.copy()` adicional? Por dos razones:

1. **Defensivo**: si más adelante hacés `melb_work.loc[...] = ...` sobre el resultado de un `.drop()` encadenado a un `.dropna()`, pandas puede tirarte `SettingWithCopyWarning` porque internamente puede haber referencias al original. Un `.copy()` explícito blinda contra eso.
2. **Pedagógico**: en EyCD la cátedra grita *"NUNCA accionar sobre el dataset original"* (cita textual). Hacer `.copy()` explícito es una señal de que entendiste la regla.

¿Se entiende la decisión? La próxima celda es trivial; la justificación es lo que importa.

In [11]:
melb_work = melb_df.drop(columns=['BuildingArea', 'YearBuilt']).copy()

print(f"Shape original:  {melb_df.shape}")
print(f"Shape sin BA/YB: {melb_work.shape}")
melb_work.head(3)


Shape original:  (13580, 21)
Shape sin BA/YB: (13580, 19)


,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,2.0,1.0,1.0,202.0,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,2.0,1.0,0.0,156.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,3.0,2.0,0.0,134.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0


### Ejercicio 1.2 — Limpieza de filas y columnas por faltantes / redundancia

**Qué pide la consigna**: "Decidir si elimina algunas filas o columnas en base al análisis de datos faltantes".

La consigna habla SOLO de faltantes, pero acá es donde la cátedra te quiere ver tomando decisiones con **criterio de curador**. No hay una respuesta única correcta; hay una respuesta justificada y otra que parece copy-paste de tutorial.

Vamos columna por columna explicando por qué cada decisión vale lo que vale.

#### Decisión 1: eliminar `Address`

Address es texto libre con **13.378 valores únicos en 13.580 filas**. Eso quiere decir que **prácticamente cada fila tiene su propio Address**. ¿Qué pasaría si lo dejamos para encodear?

- **OHE con 13.378 categorías**: la matriz post-encoding tendría 13.378 columnas binarias casi todas con un único 1 cada una. Memoria explota (~1.4 GB en sparse, peor en denso), y semánticamente cada Address pasa a ser un identificador único.
- **Ordinal encoding**: imposible: Address no tiene un orden natural ("85 Turner St" no es "menos que" "25 Bloomburg St").
- **Frecuencia**: cada Address aparece 1 o 2 veces, frecuencia es casi constante. No discrimina.

¿Hay información ÚTIL en Address más allá del identificador? Sí: el nombre de calle podría capturar zona socioeconómica. Pero ya tenemos `Suburb`, `Postcode`, `CouncilArea`, `Regionname`, `Lattitude` y `Longtitude` cubriendo esa dimensión geográfica. **Address es redundante con esas seis columnas combinadas**. Lo eliminamos.

#### Decisión 2: eliminar `Bedroom2`

En el EDA de Clase 3 (apunte `07-exploracion-eda.md`) vimos la matriz de correlación. `Bedroom2` tiene **r = 0.94** con `Rooms`. La descripción del dataset dice que `Bedroom2` viene de una fuente distinta, pero mide casi lo mismo.

¿Qué problema introduce mantenerla?

- **Multicolinealidad**: dos variables muy correlacionadas duplican señal. En regresión lineal, los coeficientes se vuelven inestables (pequeños cambios en datos cambian mucho los coeficientes). En modelos basados en distancias, las casas con `Rooms=3, Bedroom2=3` quedan ponderadas el doble en esa dimensión.
- **PCA inflado**: PCA va a poner una de sus primeras componentes capturando justamente la dirección Rooms-Bedroom2. Es "varianza fácil" que no te dice nada nuevo.
- **Costo de mantener**: nada — no te sirve, te sesga. Decisión obvia: eliminar.

¿Por qué eliminar `Bedroom2` y no `Rooms`? Porque `Rooms` es la variable original del dataset (limpia, sin faltantes); `Bedroom2` viene de una segunda fuente con posibles inconsistencias.

#### Decisión 3: dropear las 62 filas con `Car` NaN

`Car` (cantidad de cocheras) tiene 62 NaN. Sobre 13.580 filas, es el **0.46%**. Tres opciones:

| Opción | Pro | Contra |
|--------|-----|--------|
| Dropear filas | Simple, no inventamos datos | Perdés 62 observaciones (despreciable) |
| Imputar con 0 (sin cochera) | Conservás filas, asunción razonable | Asumís semántica que la cátedra no garantiza |
| Imputar con MICE en Ej 2 | Coherente con flujo del TP | Imputar el 0.46% es overkill |

Elegimos dropear. **Argumento técnico**: con MCAR y faltante < 1%, `dropna()` es la opción menos sesgada. Imputar con 0 introduce el supuesto "NaN = sin cochera" que solo es válido si el formulario de carga distinguía "0 cocheras" de "no se preguntó". Sin saber eso, dropear es más honesto.

#### Decisión 4: `CouncilArea` — llenar NaN con `'Unknown'`

`CouncilArea` (consejo municipal) tiene **1.369 NaN (10.1%)**. Acá NO es trivial.

- **¿Dropear filas?** Perdés 10% del dataset. Caro. Y posiblemente sesgado: si CouncilArea falta más en propiedades de zonas periféricas (donde los registros administrativos son menos completos), dropear esos suburbios sesga la muestra.
- **¿Imputar con la moda?** Inventás "consejo municipal = Yarra" para casas que claramente no son de Yarra. Eso es sesgo de procesamiento (apunte `03-sesgo.md`).
- **¿Imputar con CouncilArea más cercano según Lat/Lon?** Sofisticado, pero overkill para un TP de pipeline. Y arrastra el supuesto de continuidad espacial.
- **Tratar NaN como categoría 'Unknown'**: la presencia del NaN ES información. "No se reportó el consejo municipal" puede correlacionar con "propiedad de zona rural" o "registro incompleto". Conservás esa señal.

Elegimos la última. Es una decisión que la cátedra acepta y que se justifica naturalmente: respetás los datos como vinieron.

#### Trampa típica

La trampa es eliminar `CouncilArea` entera por "tiene muchos faltantes". 10% no es mucho, y la variable correlaciona fuerte con Price (los consejos premium pagan más). Tirarla porque "es muy faltante" es perder señal por pereza. La cátedra busca que vos **justifiques** cómo lidiar con el faltante, no que lo evites.

#### Resultado esperado

Después de esta limpieza, `melb_work` queda con:
- **17 columnas** (descartamos `Address`, `Bedroom2`, `BuildingArea`, `YearBuilt`).
- **13.518 filas** (descartamos 62 por Car NaN).
- **Ningún NaN** en las columnas activas (CouncilArea fue rellenado con `'Unknown'`).

Listo para encodear en 1.4.

#### Visualizando los faltantes con `missingno`

Antes de hacer cualquier `drop`/`fillna`, vale la pena ver los faltantes con un gráfico. `missingno` es la herramienta estándar — instala con `pip install missingno`. Tres vistas útiles:

- `msno.bar(df)`: barra por columna con la cantidad de no-nulos. Te muestra de un vistazo qué columnas están completas.
- `msno.matrix(df)`: cuadrícula donde cada fila es una observación y cada columna es una variable. Los blancos son NaN. Si los blancos se agrupan en patrones (rayas horizontales), hay correlación entre faltantes.
- `msno.heatmap(df)`: heatmap de correlación de FALTANTES. Si `BuildingArea` y `YearBuilt` tienen alta correlación de presencia/ausencia (>0.7), significa que cuando una falta la otra también suele faltar — son MAR conjunto, conviene imputarlas juntas (que es lo que hacemos en Ej 2).

In [ ]:
import missingno as msno

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Barra de completitud (cantidad de no-nulos por columna)
msno.bar(melb_df, ax=axes[0], color='steelblue', fontsize=10)
axes[0].set_title('Completitud por columna (no-nulos)')

# Heatmap de correlación de faltantes
msno.heatmap(melb_df, ax=axes[1], fontsize=9)
axes[1].set_title('Correlación de patrones de NaN')

plt.tight_layout()
plt.show()


In [12]:
melb_work = melb_work.drop(columns=['Address', 'Bedroom2'])

filas_antes = len(melb_work)
melb_work = melb_work.dropna(subset=['Car']).copy()
filas_despues = len(melb_work)
print(f"Filas con Car NaN eliminadas: {filas_antes - filas_despues}")

melb_work['CouncilArea'] = melb_work['CouncilArea'].fillna('Unknown')

print(f"\nShape post-limpieza: {melb_work.shape}")
print(f"\nFaltantes restantes:")
faltantes = melb_work.isnull().sum()
print(faltantes[faltantes > 0] if (faltantes > 0).any() else "(ninguno)")


Filas con Car NaN eliminadas: 62

Shape post-limpieza: (13518, 17)

Faltantes restantes:
(ninguno)


### Ejercicio 1.3 — Análisis descriptivo + matriz numérica

**Qué pide la consigna**: "Hacer un análisis descriptivo de las variables numéricas. Todas tienen el tipo `Dtype` correcto asignado? Armar una matriz (array) sólo con las variables numéricas."

Tres cosas:
1. **Describir** las numéricas (`describe()`).
2. **Auditar Dtypes**: pandas asigna el tipo según los valores, pero **el tipo computacional no siempre coincide con el tipo estadístico**. Una columna puede ser `float64` pero conceptualmente categórica.
3. **Armar la matriz** `X_num` con las que queden como numéricas legítimas.

#### La distinción clave: tipo computacional vs tipo estadístico

| Variable | Dtype pandas | Tipo estadístico | ¿OK? |
|----------|--------------|-------------------|------|
| `Rooms` | int64 | Cuantitativa discreta | Sí |
| `Price` | float64 | Cuantitativa continua | Sí |
| `Distance` | float64 | Cuantitativa continua | Sí |
| `Bathroom`, `Car` | float64 | Cuantitativa discreta | Sí |
| `Landsize` | float64 | Cuantitativa continua | Sí |
| `Lattitude`, `Longtitude` | float64 | Cuantitativa continua | Sí |
| `Propertycount` | float64 | Cuantitativa discreta | Sí |
| **`Postcode`** | **float64** | **Categórica nominal** | **NO** |

`Postcode` es el caso clásico de "numérico que en realidad es un ID". El postcode 3000 y el 3100 no están "100 unidades alejados" en ningún sentido relevante — son etiquetas. Tratarlo como número significa:

- PCA va a interpretar la magnitud absoluta como información
- KNN va a usar distancia euclidiana entre postcodes como medida de similitud
- StandardScaler le va a aplicar z-score — un disparate semántico

#### Estrategia: exploración cuantitativa de las 3 opciones para Postcode

- **A**: Postcode como numérico (la opción "perezosa")
- **B**: Postcode categórico con OHE de las 30 más frecuentes + 'Otros'
- **C**: Postcode categórico con OHE de las 198

Para cada una medimos: cantidad de columnas, memoria, varianza acumulada en 5 PCs.

In [ ]:
print("=== DESCRIBE NUMÉRICAS ===")
print(melb_work.describe().round(2))
print("\n=== DTYPES ===")
print(melb_work.dtypes)
print("\n=== CARDINALIDAD DE CATEGÓRICAS Y POSTCODE ===")
cats_y_postcode = ['Suburb', 'Type', 'Method', 'SellerG', 'Date',
                   'CouncilArea', 'Regionname', 'Postcode']
for c in cats_y_postcode:
    print(f"  {c:15} -> {melb_work[c].nunique():>5} valores únicos")


#### Lo que ves arriba

- `describe()` confirma rangos sanos en Rooms (1-10), Bathroom (1-8), Car (0-10). Pero `Landsize` tiene min=0 (sospechoso) y un max enorme (433.014 m² — un campo).
- `Postcode` tiene **198 valores únicos** — confirma que es categórico.
- `Suburb` tiene **314 valores únicos** — alta cardinalidad.
- `Date` tiene **58 valores únicos** — strings de fecha, se tratan en 1.4.
- `SellerG` tiene **268 valores únicos** — alta cardinalidad.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

NUM_BASE = ['Rooms', 'Price', 'Distance', 'Bathroom', 'Car',
            'Landsize', 'Lattitude', 'Longtitude', 'Propertycount']

# Opción A: Postcode como numérico
X_num_A = melb_work[NUM_BASE + ['Postcode']].values

# Opción B: Postcode categórico con top-30 + 'Otros'
top30_postcodes = melb_work['Postcode'].value_counts().head(30).index
postcode_reducido = melb_work['Postcode'].where(
    melb_work['Postcode'].isin(top30_postcodes), other=-1.0
)
postcode_dummies_B = pandas.get_dummies(postcode_reducido.astype(str), prefix='Postcode').astype(float).values
X_num_B_base = melb_work[NUM_BASE].values
X_num_B = numpy.hstack([X_num_B_base, postcode_dummies_B])

# Opción C: Postcode categórico SIN reducción
postcode_dummies_C = pandas.get_dummies(melb_work['Postcode'].astype(str), prefix='Postcode').astype(float).values
X_num_C = numpy.hstack([X_num_B_base, postcode_dummies_C])

def medir(X, nombre):
    Xs = StandardScaler().fit_transform(X)
    pca = PCA(n_components=5)
    pca.fit(Xs)
    return {
        'opcion': nombre,
        'shape': X.shape,
        'memoria_MB': round(X.nbytes / 1024 / 1024, 2),
        'var_5_PCs': round(pca.explained_variance_ratio_.sum(), 4),
        'var_PC1': round(pca.explained_variance_ratio_[0], 4),
    }

import pprint
for r in [
    medir(X_num_A, 'A: Postcode numérico'),
    medir(X_num_B, 'B: Postcode top-30 + Otros'),
    medir(X_num_C, 'C: Postcode 198 categorías'),
]:
    pprint.pprint(r)


#### Decisión

Vamos con la **Opción B**: Postcode como categórico, top-30 + 'Otros'. Justificación:

1. Conceptualmente correcto: tratamos un ID como categoría, no como magnitud.
2. Pragmático: 30 columnas extra es manejable.
3. Top-30 cubre ~60% del dataset.

`X_num` queda con **9 columnas** estrictamente numéricas. Postcode pasa a categóricas en Ej 1.4.

In [ ]:
COLS_NUM = ['Rooms', 'Price', 'Distance', 'Bathroom', 'Car',
            'Landsize', 'Lattitude', 'Longtitude', 'Propertycount']
COLS_CAT = ['Suburb', 'Type', 'Method', 'SellerG', 'Date',
            'CouncilArea', 'Regionname', 'Postcode']

X_num = melb_work[COLS_NUM].values
print(f"X_num shape: {X_num.shape}")
print(f"X_num dtype: {X_num.dtype}")
print(f"Memoria: {X_num.nbytes / 1024 / 1024:.2f} MB")
print(f"\nNaN en X_num: {numpy.isnan(X_num).sum()}")


### Ejercicio 1.4 — Encoding de variables categóricas

| Variable | Cardinalidad | Estrategia |
|----------|--------------|------------|
| `Type` | 3 | OHE directo |
| `Method` | 9 | OHE directo |
| `Regionname` | 8 | OHE directo |
| `CouncilArea` | 33 + 'Unknown' | OHE directo |
| `Postcode` | 198 | Reducción + OHE |
| `SellerG` | 268 | Reducción + OHE |
| `Suburb` | 314 | Reducción + OHE |
| `Date` | 58 (strings) | Convertir primero |

Plan: (1) `Date` → 4 estrategias, (2) reducción de cardinalidad → 3 estrategias, (3) OHE final.

In [ ]:
print("=== CARDINALIDAD ===\n")
for c in COLS_CAT:
    if c == 'Date':
        continue
    print(f"{c:15} | {melb_work[c].nunique():>5} únicos | top-5: {list(melb_work[c].value_counts().head(5).index)}")

print("\n=== COBERTURA TOP-N ===")
for c in ['Suburb', 'SellerG', 'Postcode']:
    top30 = melb_work[c].value_counts().head(30)
    cobertura = top30.sum() / len(melb_work)
    print(f"{c:15} | top-30 cubre {cobertura*100:.1f}% de las filas")


#### Exploración de `Date` — 4 estrategias

| Opción | Features | Captura | Costo |
|--------|----------|---------|-------|
| **A** — Descartar | 0 | Nada | Mínimo |
| **B** — Año + Mes | 2 | Tendencia + estacionalidad lineal | Bajo |
| **C** — Días desde 2016-01-01 | 1 | Tendencia continua | Bajo |
| **D** — Año + Mes cíclico (seno/coseno) | 3 | Tendencia + estacionalidad CÍCLICA | Medio |

Por qué D vale la pena pensar: el mes 12 (diciembre) y el mes 1 (enero) están adyacentes en la realidad pero numéricamente quedan máximamente alejados. Con encoding cíclico quedan cerca. Métrica: correlación con Price.

In [ ]:
melb_work['Date_dt'] = pandas.to_datetime(melb_work['Date'], format='%d/%m/%Y')

year_B = melb_work['Date_dt'].dt.year
month_B = melb_work['Date_dt'].dt.month
date_min = melb_work['Date_dt'].min()
days_C = (melb_work['Date_dt'] - date_min).dt.days
year_D = melb_work['Date_dt'].dt.year
month_num = melb_work['Date_dt'].dt.month
sin_month_D = numpy.sin(2 * numpy.pi * month_num / 12)
cos_month_D = numpy.cos(2 * numpy.pi * month_num / 12)

print("=== CORRELACIÓN DE FEATURES DERIVADAS DE Date CON Price ===\n")
print(f"Opción B - year:        {numpy.corrcoef(year_B, melb_work['Price'])[0,1]:+.4f}")
print(f"Opción B - month:       {numpy.corrcoef(month_B, melb_work['Price'])[0,1]:+.4f}")
print(f"Opción C - days:        {numpy.corrcoef(days_C, melb_work['Price'])[0,1]:+.4f}")
print(f"Opción D - sin(month):  {numpy.corrcoef(sin_month_D, melb_work['Price'])[0,1]:+.4f}")
print(f"Opción D - cos(month):  {numpy.corrcoef(cos_month_D, melb_work['Price'])[0,1]:+.4f}")


#### Decisión sobre `Date`

Las correlaciones de Date con Price son uniformemente bajas (|r| < 0.05). Date tiene baja señal predictiva.

**Elegimos Opción B (año + mes)** por simplicidad. Si las correlaciones hubieran sido fuertes, D sería mejor.

#### Exploración de cardinalidad — `Suburb`, `SellerG`, `Postcode`

- **A**: Top-N + 'Otros'
- **B**: Frequency encoding
- **C**: Drop columnas

In [ ]:
from sklearn.preprocessing import OneHotEncoder

CATS_FIJAS = ['Type', 'Method', 'CouncilArea', 'Regionname']
CATS_ALTA_CARD = ['Suburb', 'SellerG', 'Postcode']

def construir_X_cat(estrategia, top_n=30):
    df_local = melb_work[CATS_FIJAS].copy()
    if estrategia == 'top_n':
        for c in CATS_ALTA_CARD:
            top = melb_work[c].value_counts().head(top_n).index
            df_local[c] = melb_work[c].where(melb_work[c].isin(top), other='Otros').astype(str)
        enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        return enc.fit_transform(df_local)
    elif estrategia == 'frequency':
        df_freq = melb_work[CATS_ALTA_CARD].apply(lambda col: col.map(col.value_counts()))
        enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        X_fijas = enc.fit_transform(melb_work[CATS_FIJAS])
        return numpy.hstack([X_fijas, df_freq.values])
    elif estrategia == 'drop':
        enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        return enc.fit_transform(melb_work[CATS_FIJAS])

X_cat_topN = construir_X_cat('top_n', top_n=30)
X_cat_freq = construir_X_cat('frequency')
X_cat_drop = construir_X_cat('drop')

year_col   = melb_work['Date_dt'].dt.year.values.reshape(-1, 1)
month_col  = melb_work['Date_dt'].dt.month.values.reshape(-1, 1)
X_num_plus = numpy.hstack([X_num, year_col, month_col])

X_total_A = numpy.hstack([X_num_plus, X_cat_topN])
X_total_B = numpy.hstack([X_num_plus, X_cat_freq])
X_total_C = numpy.hstack([X_num_plus, X_cat_drop])

def medir_completo(X, nombre, price_idx=1):
    Xs = StandardScaler().fit_transform(X)
    pca = PCA(n_components=5)
    pca.fit(Xs)
    price = X[:, price_idx]
    correlaciones = []
    for j in range(X.shape[1]):
        if j == price_idx:
            continue
        c = numpy.corrcoef(X[:, j], price)[0, 1]
        if not numpy.isnan(c):
            correlaciones.append(abs(c))
    return {
        'opcion': nombre, 'shape': X.shape,
        'mem_MB': round(X.nbytes / 1024 / 1024, 2),
        'var_5_PCs': round(pca.explained_variance_ratio_.sum(), 4),
        'max_|corr|_con_Price': round(max(correlaciones), 4),
    }

import pprint
for r in [
    medir_completo(X_total_A, 'A: Top-30 + Otros'),
    medir_completo(X_total_B, 'B: Frequency encoding'),
    medir_completo(X_total_C, 'C: Drop columnas alta-card'),
]:
    pprint.pprint(r)


#### Decisión final

**Opción A (Top-30 + 'Otros')** para Suburb, SellerG, Postcode. Conserva identidad de las categorías frecuentes y agrupa la cola larga.

#### Resumen de tratamientos

| Variable | Tratamiento |
|----------|-------------|
| `Type`, `Method`, `Regionname`, `CouncilArea` | OHE directo |
| `Postcode`, `SellerG`, `Suburb` | Top-30 + 'Otros' → OHE |
| `Date` | Descomposición en year + month |

In [ ]:
TOP_N = 30
df_cat_final = melb_work[CATS_FIJAS].copy()
for c in CATS_ALTA_CARD:
    top = melb_work[c].value_counts().head(TOP_N).index
    df_cat_final[c] = melb_work[c].where(melb_work[c].isin(top), other='Otros').astype(str)

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat = ohe.fit_transform(df_cat_final)

print(f"X_cat shape: {X_cat.shape}")
print(f"Memoria X_cat: {X_cat.nbytes / 1024 / 1024:.2f} MB")
print(f"\nCategorías encodeadas:")
for col, cats in zip(df_cat_final.columns, ohe.categories_):
    print(f"  {col:15} -> {len(cats)} categorías")
print(f"\nTotal columnas OHE: {X_cat.shape[1]}")


### Ejercicio 1.5 — Concatenación de matriz numérica + categórica

Concatenamos con `numpy.hstack`. Orden: `[X_num | year | month | X_cat]`. Total esperado: ~155 columnas (11 numéricas + ~144 OHE).

In [ ]:
year_col  = melb_work['Date_dt'].dt.year.values.reshape(-1, 1).astype(float)
month_col = melb_work['Date_dt'].dt.month.values.reshape(-1, 1).astype(float)

X_num_final = numpy.hstack([X_num, year_col, month_col])
print(f"X_num_final shape: {X_num_final.shape}  (9 originales + year + month)")

X_total = numpy.hstack([X_num_final, X_cat])
print(f"X_cat shape:       {X_cat.shape}")
print(f"X_total shape:     {X_total.shape}")
print(f"X_total dtype:     {X_total.dtype}")
print(f"Memoria:           {X_total.nbytes / 1024 / 1024:.2f} MB")
print(f"NaN en X_total:    {numpy.isnan(X_total).sum()}")

nombres_num = list(COLS_NUM) + ['year', 'month']
nombres_cat = list(ohe.get_feature_names_out(df_cat_final.columns))
nombres_total = nombres_num + nombres_cat
print(f"\nTotal nombres de columnas: {len(nombres_total)}")


## Ejercicio 2: Imputación por KNN

En el teórico se presentó el método `IterativeImputer`, entre otros, para imputar valores faltantes en variables numéricas. Sin embargo, los ejemplos presentados sólo utilizaban algunas variables numéricas presentes en el conjunto de datos. En este ejercicio, utilizaremos la matriz de datos codificada para imputar datos faltantes de manera más precisa.

1. Agregue a la matriz obtenida en el punto anterior las columnas `YearBuilt` y `BuildingArea`.
2. Aplique una instancia de `IterativeImputer` con un estimador `KNeighborsRegressor` para imputar los valores de las variables. ¿Es necesario estandarizar o escalar los datos previamente?
3. Realice un gráfico mostrando la distribución de cada variable antes de ser imputada, y con ambos métodos de imputación.

### Ejercicio 2 — Imputación con KNN/MICE: explicación detallada

**Por qué `IterativeImputer` y no `KNNImputer` directo**:

`KNNImputer` reemplaza cada NaN por una agregación de los k vecinos más cercanos. Una pasada, sin iteración.

`IterativeImputer` (MICE — Multiple Imputation by Chained Equations):
1. Inicializa cada NaN con la mediana.
2. Para cada feature con faltantes, ajusta un modelo predictivo (KNeighborsRegressor) usando las otras features.
3. Predice los faltantes con ese modelo.
4. Itera hasta convergencia.

Cada faltante se predice usando TODA la información disponible.

#### ¿Estandarizar antes? SÍ, SÍ Y SÍ

`KNeighborsRegressor` calcula distancias euclidianas. Si Price está en cientos de miles y Rooms en {1..10}, la distancia entre dos casas va a ser dominada por la diferencia en Price. Estandarizar pone todas las features en la misma escala.

#### Comparación con baseline (mediana simple)

Vamos a aplicar DOS imputadores:
- **Mediana**: pico artificial en el valor de la mediana, destruye varianza.
- **KNN/MICE**: preserva la forma de la distribución original.

Y graficamos las distribuciones para ver visualmente el efecto.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.neighbors import KNeighborsRegressor

indices_validos = melb_work.index
ba_yb = melb_df.loc[indices_validos, ['BuildingArea', 'YearBuilt']].values

print(f"X_total previo:    {X_total.shape}")
print(f"BA/YB a agregar:   {ba_yb.shape}")
print(f"NaN en BA: {numpy.isnan(ba_yb[:, 0]).sum()} ({numpy.isnan(ba_yb[:, 0]).mean()*100:.1f}%)")
print(f"NaN en YB: {numpy.isnan(ba_yb[:, 1]).sum()} ({numpy.isnan(ba_yb[:, 1]).mean()*100:.1f}%)")

X_con_nan = numpy.hstack([X_total, ba_yb])
nombres_con_nan = nombres_total + ['BuildingArea', 'YearBuilt']
print(f"\nX_con_nan shape: {X_con_nan.shape}")
print(f"Total NaN:       {numpy.isnan(X_con_nan).sum()}")


In [ ]:
medias = numpy.nanmean(X_con_nan, axis=0)
desvios = numpy.nanstd(X_con_nan, axis=0)
desvios[desvios == 0] = 1.0

X_estandarizado = (X_con_nan - medias) / desvios

imputer_median = SimpleImputer(strategy='median')
X_imp_median_std = imputer_median.fit_transform(X_estandarizado)

imputer_knn = IterativeImputer(
    estimator=KNeighborsRegressor(n_neighbors=5),
    max_iter=10, random_state=42, verbose=0,
)
X_imp_knn_std = imputer_knn.fit_transform(X_estandarizado)

X_imp_median = X_imp_median_std * desvios + medias
X_imp_knn    = X_imp_knn_std * desvios + medias

print("=== VERIFICACIÓN ===")
print(f"NaN restantes (mediana): {numpy.isnan(X_imp_median).sum()}")
print(f"NaN restantes (KNN):     {numpy.isnan(X_imp_knn).sum()}")
print(f"\nMedia BuildingArea original: {numpy.nanmean(ba_yb[:, 0]):.2f}")
print(f"Media BA con mediana:        {X_imp_median[:, -2].mean():.2f}")
print(f"Media BA con KNN:            {X_imp_knn[:, -2].mean():.2f}")


#### Lectura: comparación visual

Tres curvas: **original** (solo filas SIN NaN), **mediana** (toda la columna con NaN reemplazados por mediana), **KNN/MICE** (imputación inteligente).

Lo que esperás ver:
- **Mediana**: pico artificial en la mediana. Para BuildingArea (~47% NaN) el pico va a ser brutal.
- **KNN/MICE**: distribución imputada se parece a la original.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ba_orig = ba_yb[:, 0]
ba_orig_clean = ba_orig[~numpy.isnan(ba_orig)]
ax.hist(ba_orig_clean, bins=60, alpha=0.4, label='Original (sin NaN)', color='gray')
ax.hist(X_imp_median[:, -2], bins=60, alpha=0.5, label='Imputado mediana', color='red')
ax.hist(X_imp_knn[:, -2], bins=60, alpha=0.5, label='Imputado KNN/MICE', color='steelblue')
ax.set_xlim(0, 400)
ax.set_xlabel('BuildingArea (m²)')
ax.set_ylabel('Frecuencia')
ax.set_title('BuildingArea: original vs imputada')
ax.legend()

ax = axes[1]
yb_orig = ba_yb[:, 1]
yb_orig_clean = yb_orig[~numpy.isnan(yb_orig)]
ax.hist(yb_orig_clean, bins=60, alpha=0.4, label='Original (sin NaN)', color='gray')
ax.hist(X_imp_median[:, -1], bins=60, alpha=0.5, label='Imputado mediana', color='red')
ax.hist(X_imp_knn[:, -1], bins=60, alpha=0.5, label='Imputado KNN/MICE', color='steelblue')
ax.set_xlim(1850, 2020)
ax.set_xlabel('YearBuilt')
ax.set_title('YearBuilt: original vs imputada')
ax.legend()

plt.tight_layout()
plt.show()

X_final = X_imp_knn
print(f"\nX_final = X_imp_knn con shape {X_final.shape} para el Ejercicio 3.")


## Ejercicio 3: Reducción de dimensionalidad.

Utilizando la matriz obtenida en el ejercicio anterior:
1. Aplique `PCA` para obtener $n$ componentes principales de la matriz, donde `n = min(20, X.shape[0])`. ¿Es necesario estandarizar o escalar los datos? Puede decidir si hacer PCA sobre todas las variables o bien seleccionar un subconjunto que crea pertinente.

2. Seleccione las proyecciones de los datos sobre las dos primeras componentes principales (las primeras dos columnas del resultado) para agregar como nuevas características al conjunto de datos.

### Ejercicio 3 — PCA: explicación detallada

#### El typo de la consigna

La consigna dice `n = min(20, X.shape[0])`. Con `X.shape = (13518, 157)`, `min(20, 13518) = 20`.

Pero conceptualmente es raro: `X.shape[0]` son filas (muestras), `X.shape[1]` son features. PCA reduce features, no muestras. Lo natural sería `n = min(20, X.shape[1])`. Es probable que sea un typo. En este caso ambas interpretaciones dan 20, pero conviene mencionarlo.

#### ¿Estandarizar antes de PCA? Sí, absolutamente

PCA captura varianza. Sin escalar, la columna con mayor varianza absoluta domina. Apunte `06-pca.md` lo dice textual de la cátedra.

#### Agregar PC1 y PC2 como features

No reemplaza X, las SUMA. PCA hace compresión óptima — las 2 primeras componentes son resúmenes ricos.

In [ ]:
from sklearn.decomposition import PCA

medias_final = X_final.mean(axis=0)
desvios_final = X_final.std(axis=0)
desvios_final[desvios_final == 0] = 1.0
X_final_std = (X_final - medias_final) / desvios_final

n_components = min(20, X_final.shape[0])
print(f"PCA con n_components = min(20, X.shape[0]) = {n_components}")
print(f"Nota: si la consigna era X.shape[1], hubiera sido min(20, {X_final.shape[1]}) = {min(20, X_final.shape[1])}")
print(f"En este caso ambas dan {n_components}.\n")

pca = PCA(n_components=n_components, random_state=42)
X_pca = pca.fit_transform(X_final_std)

print(f"X_pca shape: {X_pca.shape}")
print(f"\n=== VARIANZA EXPLICADA POR COMPONENTE ===")
for i, (ratio, acum) in enumerate(zip(pca.explained_variance_ratio_,
                                       numpy.cumsum(pca.explained_variance_ratio_))):
    print(f"  PC{i+1:2d}: {ratio*100:5.2f}%  (acumulada: {acum*100:5.2f}%)")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
xs = numpy.arange(1, n_components + 1)
ax.bar(xs, pca.explained_variance_ratio_ * 100, alpha=0.5, label='Por componente')
ax.plot(xs, numpy.cumsum(pca.explained_variance_ratio_) * 100, 'o-', color='darkred', label='Acumulada')
ax.set_xticks(xs)
ax.set_xlabel('Componente principal')
ax.set_ylabel('Varianza explicada (%)')
ax.set_title('Scree plot — varianza explicada por PCA')
ax.axhline(80, color='gray', linestyle='--', alpha=0.5, label='80%')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


#### Agregar PC1 y PC2 al conjunto de features

Las dos primeras PCs como features adicionales (no reemplazo de las 157 originales).

In [ ]:
pc1 = X_pca[:, 0].reshape(-1, 1)
pc2 = X_pca[:, 1].reshape(-1, 1)

X_con_pca = numpy.hstack([X_final, pc1, pc2])
nombres_con_pca = nombres_con_nan + ['PC1', 'PC2']

print(f"X_final shape:    {X_final.shape}")
print(f"X_con_pca shape:  {X_con_pca.shape}  (+2 PCs)")
print(f"Total nombres:    {len(nombres_con_pca)}")


## Ejercicio 4: Composición del resultado

Transformar nuevamente el conjunto de datos procesado en un `pandas.DataFrame` y guardarlo en un archivo.

Para eso, será necesario recordar el nombre original de cada columna de la matriz, en el orden correcto. Tener en cuenta:
1. El método `OneHotEncoder.get_feature_names` o el atributo `OneHotEncoder.categories_` permiten obtener una lista con los valores de la categoría que le corresponde a cada índice de la matriz.
2. Ninguno de los métodos aplicados intercambia de lugar las columnas o las filas de la matriz.

### Ejercicio 4 — Composición del resultado

Reconstruimos el DataFrame con nombres, guardamos a CSV, y **verificamos post-guardado** releyendo el archivo y validando con assertions.

In [ ]:
df_final = pandas.DataFrame(X_con_pca, columns=nombres_con_pca)
print(f"DataFrame final shape: {df_final.shape}")
print(f"\nPrimeras 5 columnas: {list(df_final.columns[:5])}")
print(f"Últimas 5 columnas:  {list(df_final.columns[-5:])}")
df_final.head(3)


In [ ]:
output_path = 'melb_data_procesado.csv'
df_final.to_csv(output_path, index=False)
print(f"Dataset guardado en: {output_path}")

import os
size_mb = os.path.getsize(output_path) / 1024 / 1024
print(f"Tamaño: {size_mb:.2f} MB")
print(f"Filas: {len(df_final)}, Columnas: {df_final.shape[1]}")


#### Verificación post-guardado

Buena práctica defensiva: releer el CSV y verificar que coincide con lo que guardamos. Si hay un bug de serialización, lo cazamos acá.

In [ ]:
df_verif = pandas.read_csv(output_path)

print(f"Shape original:    {df_final.shape}")
print(f"Shape al releer:   {df_verif.shape}")
print(f"NaN al releer:     {df_verif.isna().sum().sum()}")
print(f"Columnas idénticas: {list(df_verif.columns) == list(df_final.columns)}")

diff_max = numpy.abs(df_verif.values - df_final.values).max()
print(f"Diferencia máxima entre original y releído: {diff_max:.6e}")

assert df_verif.shape == df_final.shape, "Shape no coincide!"
assert df_verif.isna().sum().sum() == 0, "Aparecieron NaN al releer!"
print("\n[OK] Verificación pasada")


## Ejercicio 5: Documentación

En un documento `.pdf` o `.md` realizar un reporte de las operaciones que realizaron para obtener el conjunto de datos final. Se debe incluir:
  1. Criterios de exclusión (o inclusión) de filas o columnas
  2. Interpretación de las columnas presentes
  2. Todas las transofrmaciones realizadas

Este documento es de uso técnico exclusivamente, y su objetivo es permitir que otres desarrolladores puedan reproducir los mismos pasos y obtener el mismo resultado. Debe ser detallado pero consiso. Por ejemplo:

```
  ## Criterios de exclusión de ejemplos
  1. Se eliminan ejemplos donde el año de construcción es previo a 1900

  ## Características seleccionadas
  ### Características categóricas
  1. Type: tipo de propiedad. 3 valores posibles
  2. ...
  Todas las características categóricas fueron codificadas con un
  método OneHotEncoding utilizando como máximo sus 30 valores más
  frecuentes.
  
  ### Características numéricas
  1. Rooms: Cantidad de habitaciones
  2. Distance: Distancia al centro de la ciudad.
  3. airbnb_mean_price: Se agrega el precio promedio diario de
     publicaciones de la plataforma AirBnB en el mismo código
     postal. [Link al repositorio con datos externos].

  ### Transformaciones:
  1. Todas las características numéricas fueron estandarizadas.
  2. La columna `Suburb` fue imputada utilizando el método ...
  3. Las columnas `YearBuilt` y ... fueron imputadas utilizando el
     algoritmo ...
  4. ...

  ### Datos aumentados
  1. Se agregan las 5 primeras columnas obtenidas a través del
     método de PCA, aplicado sobre el conjunto de datos
     totalmente procesado.
```


### Ejercicio 5 — Documentación reproducible

Generamos `reporte_tp1.md` desde el propio notebook (con pipeline ASCII para visualización rápida).

In [ ]:
# Generar reporte como archivo .md
from pathlib import Path

pipeline_ascii = """
Dataset original (13.580 x 21)
  |
  +-- Ej 1.1 -> drop BuildingArea, YearBuilt (~47% y ~40% faltantes)
  |             -> (13.580 x 19)
  |
  +-- Ej 1.2 -> drop Address (cardinalidad casi unica)
  |             drop Bedroom2 (r=0.94 con Rooms)
  |             drop 62 filas con Car NaN (0.46%)
  |             fillna CouncilArea con 'Unknown'
  |             -> (13.518 x 17)
  |
  +-- Ej 1.3 -> describe + dtype audit
  |             Postcode reclasificado como CATEGORICO
  |             X_num = matriz numerica (9 columnas)
  |
  +-- Ej 1.4 -> Date -> year + month (numericas)
  |             Suburb / SellerG / Postcode: top-30 + 'Otros' -> OHE
  |             Type, Method, CouncilArea, Regionname: OHE directo
  |             X_cat: (13.518 x ~144)
  |
  +-- Ej 1.5 -> np.hstack([X_num, year, month, X_cat])
  |             X_total: (13.518 x ~155)
  |
  +-- Ej 2   -> agregar BuildingArea + YearBuilt (con NaN)
  |             StandardScaler -> IterativeImputer(KNN k=5)
  |             Comparacion visual vs SimpleImputer(median)
  |             X_final: (13.518 x ~157)
  |
  +-- Ej 3   -> StandardScaler + PCA(n=20)
  |             [Typo consigna: X.shape[0] vs X.shape[1]]
  |             Agregar PC1, PC2 como features
  |             X_con_pca: (13.518 x ~159)
  |
  +-- Ej 4   -> reconstruccion a DataFrame con nombres
  |             guardar a CSV + verificacion post-guardado
  |             melb_data_procesado.csv
  |
  +-- Ej 5   -> reporte_tp1.md (este archivo)
"""

reporte = f"""# Reporte tecnico - TP1 (DiploDatos 2026, EyCD)

**Dataset original**: melb_data.csv (13.580 x 21).
**Dataset final**: melb_data_procesado.csv (13.518 x ~159).
**Reduccion de filas**: 62 (filas con Car NaN dropeadas).

## Pipeline completo
```{pipeline_ascii}```

## 1. Criterios de exclusion de columnas

| Columna | Accion | Justificacion |
|---------|--------|---------------|
| BuildingArea | Excluida del encoding inicial, reincorporada en Ej 2 | 47% faltantes. Imputarla con info encodeada. |
| YearBuilt | Excluida del encoding inicial, reincorporada en Ej 2 | 40% faltantes. |
| Address | Eliminada | 13.378 unicos / 13.580 filas. Identificador, redundante. |
| Bedroom2 | Eliminada | r=0.94 con Rooms. Multicolinealidad. |

## 2. Criterios de exclusion de filas

62 filas con Car NaN dropeadas (0.46%, perdida despreciable).

## 3. Tratamiento de faltantes

1.369 filas con CouncilArea NaN: rellenadas con 'Unknown' como categoria.

## 4. Caracteristicas conservadas

### Numericas (11)
Rooms, Price, Distance, Bathroom, Car, Landsize, Lattitude, Longtitude, Propertycount, year, month.

### Categoricas (OHE)
- Type, Method, Regionname, CouncilArea: OHE directo.
- Suburb, SellerG, Postcode: top-30 + 'Otros' -> OHE.

### Imputadas
BuildingArea, YearBuilt con IterativeImputer(KNeighborsRegressor(k=5)) + StandardScaler previo.

### PCA
PC1, PC2 agregadas como features.

## 5. Transformaciones

1. Encoding categorico con OneHotEncoder(handle_unknown='ignore').
2. Date -> year + month (correlaciones <0.05 con Price descartan estrategias mas complejas).
3. Imputacion con IterativeImputer(KNeighborsRegressor) + StandardScaler.
4. PCA con StandardScaler + PCA(n=20).

## 6. Decisiones de criterio

- Postcode como categorica (no numerica): es un ID.
- CouncilArea con 'Unknown': la ausencia es informacion.
- Bedroom2 eliminada: redundancia con Rooms.
- Reduccion top-30: balance entre identidad y dimensionalidad.
- Imputacion KNN vs mediana: KNN preserva distribucion.

## 7. Observaciones criticas a la consigna

Ej 3: n = min(20, X.shape[0]) probablemente sea typo por X.shape[1]. En este caso ambos dan 20.

## 8. Estado final

- Archivo: melb_data_procesado.csv (13.518 x ~159).
- Sin NaN.
- Verificado post-guardado.
"""

reporte_path = Path('reporte_tp1.md')
reporte_path.write_text(reporte, encoding='utf-8')
print(f"Reporte guardado en: {reporte_path.resolve()}")
print(f"Tamaño: {reporte_path.stat().st_size} bytes")
